# 1. Constants, packages and path

In [133]:
# constants for LPG supply chain processing
default_LPG_price = 0.6  # USD/kg
STS_cost = 0.1  # 10%
pre_bottling_cost = 0.2  # 20% TO BE MODIFIED
CRUDE_BARREL_TO_TONS = 7.33  # barrel of crude per ton
BUFFER_KM = 10  # km radius for spatial buffer queries

import pandas as pd
import geopandas as gpd
from pathlib import Path
import numpy as np
import rasterio
from rasterio.mask import mask as raster_mask
from shapely.geometry import Point, box

# Set up data path
data_dir = Path("dataset_first_step")
data_dir.mkdir(exist_ok=True)

# 2. create mask layer (just to try)

In [134]:
# Build mask_layer from friction raster footprint
from rasterio.features import shapes
from shapely.geometry import shape
from shapely.ops import unary_union

friction_mask_path = data_dir / "friction_moto.tif"
if not friction_mask_path.exists():
    raise FileNotFoundError(f"Missing friction raster for mask creation: {friction_mask_path}")

with rasterio.open(friction_mask_path) as src:
    if src.crs is None:
        raise ValueError("friction_moto.tif has no CRS")

    # Valid data footprint: non-nodata pixels
    valid_mask = src.dataset_mask() > 0
    if not np.any(valid_mask):
        raise ValueError("friction_moto.tif contains no valid pixels to build mask_layer")

    mask_shapes = [
        shape(geom)
        for geom, value in shapes(valid_mask.astype(np.uint8), mask=valid_mask, transform=src.transform)
        if value == 1
    ]

    if len(mask_shapes) == 0:
        raise ValueError("Could not extract a valid geometry footprint from friction_moto.tif")

    mask_geom = unary_union(mask_shapes)
    mask_layer = gpd.GeoDataFrame(geometry=[mask_geom], crs=src.crs).to_crs("EPSG:4326")

print(f"Mask layer initialized from friction raster ({len(mask_layer)} geometry)")
print("Constants declared and imports completed")

Mask layer initialized from friction raster (1 geometry)
Constants declared and imports completed


# 3. Functions: data loading and fill missing values

In [135]:
# Load input data files with smart defaults
from onstove.layer import VectorLayer, RasterLayer
from shapely.ops import nearest_points

# Define default gpkg and layer names
DEFAULT_GPKG = data_dir / "default.gpkg"
DEFAULT_LAYERS = {
    'refineries': 'refineries',
    'ports': 'ports',
    'gas_plants': 'gas_plants',
    'primary_storage': 'primary_storage',
    'border_points': 'border_points',
}


def _as_vector_layer(gdf, name="mask"):
    vect = VectorLayer(name=name)
    vect.data = gdf.copy()
    return vect


def load_raster_means(raster_path, band_names, mask_gdf):
    """
    Read a multiband raster, clip by mask_gdf, and return band averages.

    Output format matches previous table usage:
    - index: band names
    - single numeric column
    """
    if not raster_path.exists():
        raise FileNotFoundError(f"Missing raster file: {raster_path}")

    if mask_gdf.crs is None:
        raise ValueError("mask_layer has no CRS")

    mask_vect = _as_vector_layer(mask_gdf, name="mask_layer")
    values = {}

    for band_idx, band_name in enumerate(band_names, start=1):
        raster = RasterLayer(name=f"{raster_path.stem}_{band_name}", path=str(raster_path), band=band_idx)
        raster.mask(mask_vect, crop=True, all_touched=False)

        arr = raster.data.astype(np.float64)
        nodata = raster.meta.get('nodata')
        if nodata is not None:
            try:
                if not np.isnan(nodata):
                    arr[arr == nodata] = np.nan
            except TypeError:
                arr[arr == nodata] = np.nan

        mean_val = np.nanmean(arr)
        if np.isnan(mean_val):
            raise ValueError(
                f"Band '{band_name}' in {raster_path.name} has no valid data inside mask_layer"
            )
        values[band_name] = float(mean_val)

    result = pd.DataFrame.from_dict(values, orient='index')
    result.columns = [1]
    return result


def load_or_use_default(gpkg_name, default_gpkg_path, default_layer_name, mask_gdf, data_dir=data_dir):
    """
    SMART DATA LOADING with fallback to defaults

    Two-phase strategy:
    1. If user provided data (file exists in dataset_first_step/), load it directly
    2. If not, load default layer from default.gpkg, clip it with mask_gdf, and keep it in memory
    """
    gpkg_path = data_dir / f"{gpkg_name}.gpkg"
    mask_vect = _as_vector_layer(mask_gdf, name="mask_layer")

    if gpkg_path.exists():
        # PHASE 1: User data found
        layer = VectorLayer(name=gpkg_name, path=str(gpkg_path))
        layer.mask(mask_vect, keep_geom_type=False)
        gdf = layer.data
        gdf = gdf.reset_index(drop=True)
        gdf['id_supply'] = [f"{gpkg_name}_{i}" for i in range(len(gdf))]
        print(f"{gpkg_name}: loaded and clipped in memory ({len(gdf)} records)")
        return gdf

    # PHASE 2: Use default.gpkg layer and clip with mask_layer (no file creation)
    if not default_gpkg_path.exists():
        raise FileNotFoundError(f"Missing default gpkg: {default_gpkg_path}")

    default_gdf = gpd.read_file(default_gpkg_path, layer=default_layer_name)
    default_layer = _as_vector_layer(default_gdf, name=gpkg_name)
    default_layer.mask(mask_vect, keep_geom_type=False)

    gdf = default_layer.data
    gdf = gdf.reset_index(drop=True)
    gdf['id_supply'] = [f"{gpkg_name}_{i}" for i in range(len(gdf))]
    print(f"{gpkg_name}: clipped from default.gpkg/{default_layer_name} in memory ({len(gdf)} records)")
    return gdf


def load_border_points_or_default(mask_gdf, data_dir=data_dir, near_km=5):
    """
    Load border_points from user file if provided, otherwise from default.gpkg.

    Spatial rule:
    - Keep points inside mask_layer as-is.
    - If outside but within near_km from mask_layer, snap to nearest point on mask boundary.
    - Drop points farther than near_km from mask_layer.
    """
    user_path = data_dir / "border_points.gpkg"

    if user_path.exists():
        try:
            points = gpd.read_file(user_path, layer="border_points")
        except Exception:
            points = gpd.read_file(user_path)
        source_name = f"{user_path.name}"
    else:
        if not DEFAULT_GPKG.exists():
            raise FileNotFoundError(f"Missing default gpkg: {DEFAULT_GPKG}")
        points = gpd.read_file(DEFAULT_GPKG, layer=DEFAULT_LAYERS['border_points'])
        source_name = f"default.gpkg/{DEFAULT_LAYERS['border_points']}"

    if points.empty:
        print(f"border_points: loaded 0 records from {source_name}")
        points = points.reset_index(drop=True)
        points['id_supply'] = [f"border_points_{i}" for i in range(len(points))]
        return points

    if mask_gdf.crs is None:
        raise ValueError("mask_layer has no CRS")
    if points.crs is None:
        raise ValueError("border_points has no CRS")

    pts = points.to_crs(mask_gdf.crs).copy()

    pts_m = pts.to_crs("EPSG:3857").copy()
    mask_m = mask_gdf.to_crs("EPSG:3857")
    mask_union_m = mask_m.geometry.unary_union

    inside_count = 0
    moved_count = 0
    dropped_count = 0

    keep_rows = []
    new_geom_m = []

    for idx, geom_m in zip(pts_m.index, pts_m.geometry):
        if geom_m is None or geom_m.is_empty:
            dropped_count += 1
            continue

        if mask_union_m.covers(geom_m):
            inside_count += 1
            keep_rows.append(idx)
            new_geom_m.append(geom_m)
            continue

        dist_m = float(geom_m.distance(mask_union_m))
        if dist_m <= near_km * 1000:
            snapped = nearest_points(geom_m, mask_union_m)[1]
            moved_count += 1
            keep_rows.append(idx)
            new_geom_m.append(snapped)
        else:
            dropped_count += 1

    if len(keep_rows) == 0:
        out = pts.iloc[0:0].copy()
    else:
        out_m = pts_m.loc[keep_rows].copy()
        out_m.geometry = new_geom_m
        out = out_m.to_crs(mask_gdf.crs)

    out = out.reset_index(drop=True)
    out['id_supply'] = [f"border_points_{i}" for i in range(len(out))]
    print(
        f"border_points: loaded from {source_name} | inside={inside_count}, moved_to_mask={moved_count}, dropped_far={dropped_count}"
    )
    return out


def fill_from_buffer(gdf, default_gdf, cols, buffer_km=BUFFER_KM):
    """
    FILL MISSING VALUES using spatial buffer queries

    For each geometry in gdf with NaN values in 'cols', search for the nearest value
    in default_gdf within a radius of buffer_km.
    """
    gdf, default_gdf = gdf.copy(), default_gdf.to_crs(gdf.crs)
    buffer_dist = buffer_km / 111.0 if gdf.crs.is_geographic else buffer_km * 1000

    for col in cols:
        if col not in gdf.columns:
            continue
        missing_idx = gdf[gdf[col].isna() | (gdf[col] == '')].index
        if len(missing_idx) == 0:
            continue
        print(f"  Filling '{col}': {len(missing_idx)} rows")

        for idx in missing_idx:
            port_buffer = gdf.loc[idx, 'geometry'].buffer(buffer_dist)
            nearby = default_gdf[default_gdf.geometry.intersects(port_buffer)]

            if len(nearby) > 0 and col in nearby.columns:
                val = nearby[col].dropna()
                if len(val) > 0:
                    gdf.at[idx, col] = val.iloc[0]
    return gdf

# 4. Functions: redistribution of small national shares

In [136]:
def redistribute_small_national_shares(nat_shares_df, min_share_pct=1.0):
    """
    Redistribute national shares when a supply mode is below a minimum percentage.

    Rules:
    - Modes with share < min_share_pct are set to 0.
    - Their share is redistributed proportionally across remaining modes.
    - Final shares sum to 100.
    """
    if nat_shares_df.shape[1] == 0:
        raise ValueError("nat_shares_df has no numeric column")

    out = nat_shares_df.copy()
    share_col = out.columns[0]

    shares = pd.to_numeric(out[share_col], errors='coerce').fillna(0.0)
    keep = shares >= float(min_share_pct)

    if keep.sum() == 0:
        raise ValueError(
            f"All supply modes are below the threshold ({min_share_pct}%). Cannot redistribute."
        )

    redistributed = shares.where(keep, 0.0)
    total_keep = float(redistributed.sum())

    if total_keep <= 0:
        raise ValueError("No positive share available after threshold filtering.")

    redistributed = redistributed * (100.0 / total_keep)
    out[share_col] = redistributed
    return out


# 5. Execute data loading (or take default data)

In [137]:
# Load all data
try:
    refineries = load_or_use_default("refineries", DEFAULT_GPKG, DEFAULT_LAYERS['refineries'], mask_layer)
    ports = load_or_use_default("ports", DEFAULT_GPKG, DEFAULT_LAYERS['ports'], mask_layer)
    gas_plants = load_or_use_default("gas_plants", DEFAULT_GPKG, DEFAULT_LAYERS['gas_plants'], mask_layer)
    primary_storage = load_or_use_default("primary_storage", DEFAULT_GPKG, DEFAULT_LAYERS['primary_storage'], mask_layer)
    border_points = load_border_points_or_default(mask_layer)

    # Read multiband rasters and use average over mask_layer as actual values
    national_shares = load_raster_means(
        data_dir / "national_shares.tif",
        ['ports', 'refineries', 'gas_plants', 'border_points'],
        mask_layer,
    )
    lpg_import_cost = load_raster_means(
        data_dir / "lpg_import_cost.tif",
        ['import_land', 'import_sea'],
        mask_layer,
    )

    print("\nNational shares (mean over masked area, raw):")
    print(national_shares)

    # Redistribute shares below 1% across the remaining supply modes
    national_shares = redistribute_small_national_shares(national_shares, min_share_pct=1.0)

    print("\nNational shares (after redistribution of modes < 1%):")
    print(national_shares)

    print("\nLPG import costs (mean over masked area):")
    print(lpg_import_cost)

    # Load defaults once (reuse for all fillings)
    default_data = {name: gpd.read_file(DEFAULT_GPKG, layer=layer_name) for name, layer_name in DEFAULT_LAYERS.items()}

    print("\n✓ Data loaded successfully")
except (FileNotFoundError, ValueError) as e:
    print(f"Data loading error: {e}")

refineries: clipped from default.gpkg/refineries in memory (4 records)
ports: clipped from default.gpkg/ports in memory (11 records)
gas_plants: clipped from default.gpkg/gas_plants in memory (5 records)
primary_storage: clipped from default.gpkg/primary_storage in memory (19 records)
border_points: loaded from default.gpkg/border_points | inside=3, moved_to_mask=3, dropped_far=176


C:\Users\Fra\AppData\Local\Temp\ipykernel_21656\165749357.py:138: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The national_shares_ports layer do not have a defined nodata value, thus np.nan was assigned. You can change this defining the nodata value in the metadata of the variable as: variable.meta['nodata'] = value
d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The national_shares_refineries layer do not have a defined nodata value, thus np.nan was assigned. You can change this defining the nodata value in the metadata of the variable as: variable.meta['nodata'] = value
d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The national_shares_gas_plants layer do not have a defined nodata value, thus np.nan was assigned. You can change this defining the nodata value in the metadata of the variabl


National shares (mean over masked area, raw):
                       1
ports          50.716459
refineries      0.063389
gas_plants     49.198556
border_points   0.021596

National shares (after redistribution of modes < 1%):
                       1
ports          50.759597
refineries      0.000000
gas_plants     49.240403
border_points   0.000000

LPG import costs (mean over masked area):
                    1
import_land  0.094122
import_sea   0.019422

✓ Data loaded successfully


# 6. Functions: facility processing (fill missing values) and port cost calculation

In [138]:
# Process refineries: fill missing values and ensure key fields

def fill_lpg_price_from_raster(gdf, raster_path, band_index=1, overwrite=False):
    """
    Fill LPG_price by sampling a raster at each facility location.

    - Uses geometry centroid for non-point features.
    - If overwrite=False, only fills missing LPG_price values.
    """
    out = gdf.copy()

    if out.crs is None:
        raise ValueError("Input GeoDataFrame has no CRS")

    if 'LPG_price' not in out.columns:
        out['LPG_price'] = np.nan

    raster_file = Path(raster_path)
    if not raster_file.exists():
        raise FileNotFoundError(f"Missing LPG price raster: {raster_file}")

    with rasterio.open(raster_file) as src:
        if src.crs is None:
            raise ValueError("LPG price raster has no CRS")
        if band_index < 1 or band_index > src.count:
            raise ValueError(
                f"Requested band {band_index} for {raster_file.name}, but raster has {src.count} band(s)"
            )

        work = out.to_crs(src.crs).copy()
        geom_series = work.geometry
        if not work.geom_type.eq('Point').all():
            geom_series = work.geometry.centroid

        sampled_values = np.full(len(work), np.nan, dtype=np.float64)
        valid_positions = []
        valid_coords = []

        for pos, geom in enumerate(geom_series):
            if geom is None or geom.is_empty:
                continue
            valid_positions.append(pos)
            valid_coords.append((geom.x, geom.y))

        if len(valid_coords) > 0:
            for pos, sample in zip(valid_positions, src.sample(valid_coords, indexes=band_index)):
                sampled_values[pos] = float(sample[0])

        nodata = src.nodata
        if nodata is not None:
            sampled_values[sampled_values == nodata] = np.nan

    sampled_series = pd.Series(sampled_values, index=out.index)
    if overwrite:
        mask = sampled_series.notna()
    else:
        mask = out['LPG_price'].isna() & sampled_series.notna()

    out.loc[mask, 'LPG_price'] = sampled_series.loc[mask]
    return out


def _as_bool_series(series):
    def _to_bool(v):
        if isinstance(v, bool):
            return v
        if pd.isna(v):
            return False
        s = str(v).strip().lower()
        return s in {'true', '1', 'yes', 'y', 't'}

    return series.apply(_to_bool)


def process_refineries(gdf, default_gdf, lpg_price_raster_path, price_band=1):
    """
    REFINERIES PROCESSING

    Fill missing values from buffer and ensure key fields.

    LPG_price fallback is sampled from fob_per_kg raster at facility location.
    """
    gdf = gdf.copy()

    # Phase 1: Fill from buffer
    fill_cols = ['crude_capacity', 'LPG_price']
    gdf = fill_from_buffer(gdf, default_gdf, fill_cols)

    # Phase 2: Ensure production and fill LPG_price from raster
    if 'crude_capacity' not in gdf.columns:
        gdf['crude_capacity'] = 0.0
    else:
        gdf['crude_capacity'] = pd.to_numeric(gdf['crude_capacity'], errors='coerce').fillna(0.0)

    gdf = fill_lpg_price_from_raster(gdf, lpg_price_raster_path, band_index=price_band, overwrite=False)

    missing_prices = int(gdf['LPG_price'].isna().sum())
    if missing_prices > 0:
        print(f"Warning: {missing_prices} refinery rows still have missing LPG_price after raster sampling")

    return gdf


def process_gas_plants(gdf, default_gdf, lpg_price_raster_path, price_band=1):
    """
    GAS PLANTS PROCESSING

    Fill missing values from buffer and ensure key fields.

    LPG_price fallback is sampled from fob_per_kg raster at facility location.
    """
    gdf = gdf.copy()

    # Phase 1: Fill from buffer
    fill_cols = ['LPG_prod', 'LPG_price']
    gdf = fill_from_buffer(gdf, default_gdf, fill_cols)

    # Phase 2: Ensure production and fill LPG_price from raster
    if 'LPG_prod' not in gdf.columns:
        gdf['LPG_prod'] = 0.0
    else:
        gdf['LPG_prod'] = pd.to_numeric(gdf['LPG_prod'], errors='coerce').fillna(0.0)

    gdf = fill_lpg_price_from_raster(gdf, lpg_price_raster_path, band_index=price_band, overwrite=False)

    missing_prices = int(gdf['LPG_price'].isna().sum())
    if missing_prices > 0:
        print(f"Warning: {missing_prices} gas plant rows still have missing LPG_price after raster sampling")

    return gdf


def process_ports(gdf, default_gdf, storage_gdf, import_cost_df, lpg_price_raster_path, price_band=1):
    """
    PORTS PROCESSING

    Fill missing values, find nearest storage facility, and adjust pricing based on compliance.

    Missing LPG_price values are filled from fob_per_kg raster at each port location.
    """
    gdf = gdf.copy()

    # Phase 1: Fill missing values from buffer
    fill_cols = ['VLGC_compliance', 'LPG_price']
    gdf = fill_from_buffer(gdf, default_gdf, fill_cols)

    # Phase 2: Ensure LPG_price exists and fill from raster values
    if 'LPG_price' not in gdf.columns:
        gdf['LPG_price'] = np.nan

    gdf = fill_lpg_price_from_raster(gdf, lpg_price_raster_path, band_index=price_band, overwrite=False)

    # Phase 3: Initialize port-specific fields
    gdf['LPG_capacity'] = 0.0
    gdf['tanks_nearby'] = 0.0
    gdf['LPG_compliance'] = False

    # Ensure VLGC_compliance exists and fill NaN with False
    if 'VLGC_compliance' not in gdf.columns:
        gdf['VLGC_compliance'] = False
    else:
        gdf['VLGC_compliance'] = gdf['VLGC_compliance'].fillna(False)

    gdf = gdf.to_crs(storage_gdf.crs) if storage_gdf is not None else gdf

    # Phase 4: Find nearby storage facilities and compute de-duplicated LPG_capacity
    if storage_gdf is not None:
        buffer_dist = BUFFER_KM / 111.0 if gdf.crs.is_geographic else BUFFER_KM * 1000

        # Precompute buffers for all ports
        port_buffers = {idx: geom.buffer(buffer_dist) for idx, geom in gdf.geometry.items()}

        # Count how many ports are connected to each storage (within buffer)
        storage_connection_counts = {}
        for s_idx, s_row in storage_gdf.iterrows():
            s_geom = s_row.geometry
            n_connected_ports = sum(1 for p_buf in port_buffers.values() if p_buf.intersects(s_geom))
            storage_connection_counts[s_idx] = n_connected_ports

        # For each port, sum nearby capacities after dividing each storage by its number of connected ports
        for idx, port_buffer in port_buffers.items():
            nearby = storage_gdf[storage_gdf.geometry.intersects(port_buffer)]

            if len(nearby) > 0:
                allocated_caps = []
                for s_idx, s_row in nearby.iterrows():
                    raw_cap = float(s_row['LPG_capacity']) if ('LPG_capacity' in nearby.columns and pd.notna(s_row.get('LPG_capacity'))) else 0.0
                    n_conn = storage_connection_counts.get(s_idx, 0)
                    if n_conn > 0:
                        allocated_caps.append(raw_cap / n_conn)

                gdf.at[idx, 'LPG_capacity'] = float(sum(allocated_caps))
                gdf.at[idx, 'tanks_nearby'] = len(nearby)
                gdf.at[idx, 'LPG_compliance'] = True
            else:
                gdf.at[idx, 'LPG_capacity'] = 0.0
                gdf.at[idx, 'tanks_nearby'] = 0
                gdf.at[idx, 'LPG_compliance'] = False

    # Phase 5: Price adjustment based on LPG_compliance
    import_sea = import_cost_df.loc['import_sea'].iloc[0]

    # Ports WITH storage nearby (LPG_compliance = True)
    compliant = gdf['LPG_compliance'] == True
    gdf.loc[compliant, 'LPG_price'] = gdf.loc[compliant, 'LPG_price'] * (1 + import_sea)
    # Add STS_cost for non-VLGC compliant ports
    non_vlgc_compliant = compliant & (gdf['VLGC_compliance'] != True)
    gdf.loc[non_vlgc_compliant, 'LPG_price'] *= (1 + STS_cost)

    # Ports WITHOUT storage nearby (LPG_compliance = False)
    non_compliant = gdf['LPG_compliance'] == False
    gdf.loc[non_compliant, 'LPG_price'] = gdf.loc[non_compliant, 'LPG_price'] * (1 + import_sea) * (1 + pre_bottling_cost)

    missing_prices = int(gdf['LPG_price'].isna().sum())
    if missing_prices > 0:
        print(f"Warning: {missing_prices} port rows still have missing LPG_price after raster sampling")

    return gdf


def process_primary_storage(gdf, default_gdf, lpg_price_raster_path, price_band=1):
    """
    PRIMARY STORAGE PROCESSING

    Fill missing values from buffer and ensure key fields.
    """
    gdf = gdf.copy()

    # Phase 1: Fill from buffer for LPG_capacity 
    fill_cols = ['LPG_capacity']
    gdf = fill_from_buffer(gdf, default_gdf, fill_cols)

    # Phase 2: Ensure storage capacity
    if 'LPG_capacity' not in gdf.columns:
        gdf['LPG_capacity'] = 0.0
    else:
        gdf['LPG_capacity'] = gdf['LPG_capacity'].fillna(0.0)

    gdf['LPG_price'] = np.nan

    return gdf

def process_border_points(gdf, default_gdf, nat_shares_df, import_cost_df, lpg_price_raster_path, price_band=1):
    """
    BORDER POINTS PROCESSING

    - Fill missing LPG_price and osbp from default border_points (buffer-based)
    - Ensure LPG_price exists (sample from fob_per_kg if still missing)
    - Apply land import cost
    - Ensure osbp exists (set to 'no' if still missing)
    - Compute percentage so that sum equals national border_points share
    - If LPG_capacity exists and has positive total, use it for proportions
    - Otherwise use equal share, but osbp=true gets double weight
    """
    bp = gdf.copy()

    if bp.empty:
        bp['LPG_price'] = pd.Series(dtype='float64')
        bp['osbp'] = pd.Series(dtype='object')
        bp['percentage'] = pd.Series(dtype='float64')
        bp['category'] = pd.Series(dtype='object')
        return bp

    if 'LPG_price' not in bp.columns:
        bp['LPG_price'] = np.nan
    if 'osbp' not in bp.columns:
        bp['osbp'] = np.nan
    else:
        bp['osbp'] = bp['osbp'].replace('', np.nan)

    # Phase 1: explicit fill from default border_points for key fields
    bp = fill_from_buffer(bp, default_gdf, ['LPG_price', 'osbp'])

    # Phase 2: if LPG_price is still missing, fallback to raster sampling
    bp = fill_lpg_price_from_raster(bp, lpg_price_raster_path, band_index=price_band, overwrite=False)

    # Modifica minima: estrazione e applicazione del costo di importazione via terra
    import_land = float(import_cost_df.loc['import_land'].iloc[0])
    bp['LPG_price'] = bp['LPG_price'] * (1 + import_land)

    # Phase 3: if osbp is still missing, fallback to default 'no'
    bp['osbp'] = bp['osbp'].replace('', np.nan).fillna('no')

    border_share = float(nat_shares_df.loc['border_points'].iloc[0]) / 100.0

    if 'LPG_capacity' in bp.columns:
        cap = pd.to_numeric(bp['LPG_capacity'], errors='coerce').fillna(0.0)
        if cap.sum() > 0:
            weights = cap
        else:
            if 'osbp' in bp.columns:
                osbp_flag = _as_bool_series(bp['osbp'])
                weights = pd.Series(np.where(osbp_flag, 2.0, 1.0), index=bp.index, dtype='float64')
            else:
                weights = pd.Series(1.0, index=bp.index, dtype='float64')
    else:
        if 'osbp' in bp.columns:
            osbp_flag = _as_bool_series(bp['osbp'])
            weights = pd.Series(np.where(osbp_flag, 2.0, 1.0), index=bp.index, dtype='float64')
        else:
            weights = pd.Series(1.0, index=bp.index, dtype='float64')

    weight_sum = float(weights.sum())
    if weight_sum > 0:
        bp['percentage'] = border_share * (weights / weight_sum)
    else:
        bp['percentage'] = 0.0

    bp['category'] = 'border_points'

    if 'country' not in bp.columns:
        bp['country'] = 'Unknown'
    else:
        bp['country'] = bp['country'].fillna('Unknown')

    missing_prices = int(bp['LPG_price'].isna().sum())
    if missing_prices > 0:
        print(f"Warning: {missing_prices} border_points rows still have missing LPG_price after raster sampling")

    print(f"border_points: total percentage assigned = {bp['percentage'].sum():.2%}")
    return bp

# 7. transport costs parameters and functions

In [139]:
# TANKER COST PARAMETERS
tanker_capacity_kg = 24000
utilization_factor = 0.85
variable_cost_per_km = 0.635
driver_annual_salary_usd = 2604
salary_multiplier = 1.18
hours_per_day = 8
discount_rate = 0.03 #OnStove uses 0.03 for social perspective, 0.15 for private perspective

tanker_overnight_cost_usd = 197673 
tanker_life_years = 35
license_cost_usd = 418
licence_life_years = 5
days_per_year = 330
fixed_loading_unloading_hours = 1.0
tanker_annual_km = 42000 


# FUNCTION: CRF (Capital Recovery Factor)
def crf(rate: float, years: int) -> float:
    """Capital Recovery Factor per annualizzare costi"""
    return (rate * (1.0 + rate) ** years) / ((1.0 + rate) ** years - 1.0)


# CALCULATE FIXED COSTS
effective_load_kg = tanker_capacity_kg * utilization_factor
print(f"Effective load {100*utilization_factor:.0f}%): {effective_load_kg:.1f} kg")

driver_hourly_cost_usd = (
    driver_annual_salary_usd * salary_multiplier / (hours_per_day * days_per_year)
)
print(f"Hourly labor cost: ${driver_hourly_cost_usd:.4f}/h")

crf_tanker = crf(discount_rate, tanker_life_years)
crf_license = crf(discount_rate, licence_life_years)
annual_capital_cost_usd = tanker_overnight_cost_usd * crf_tanker
annual_license_cost_usd = license_cost_usd * crf_license
fixed_annual_tanker_cost_usd = annual_capital_cost_usd + annual_license_cost_usd

print(f"\nFixed annual costs:")
print(f"  Capital (tanker): ${annual_capital_cost_usd:,.2f}/y")
print(f"  Licence: ${annual_license_cost_usd:,.2f}/y")
print(f"  TOTAL: ${fixed_annual_tanker_cost_usd:,.2f}/y")

# FUNCTION: Fixed cost per kg
def calculate_tanker_fixed_cost_per_kg(avg_one_way_dist_km: float) -> float:
    avg_round_trip_km = 2.0 * avg_one_way_dist_km
    trips_per_year = tanker_annual_km / avg_round_trip_km
    fixed_tanker_cost_per_kg = fixed_annual_tanker_cost_usd / (effective_load_kg * trips_per_year)
    return fixed_tanker_cost_per_kg

# FUNCTION: variable cost per kg
def calculate_tanker_variable_cost_per_kg(
    one_way_distance_km: float,
    one_way_time_min: float
) -> float:
    round_trip_hours = (one_way_time_min * 2.0 / 60.0) + fixed_loading_unloading_hours
    round_trip_distance_km = 2.0 * one_way_distance_km
    variable_cost_trip = (
        variable_cost_per_km * round_trip_distance_km 
        + driver_hourly_cost_usd * round_trip_hours
    )
    variable_cost_per_kg = variable_cost_trip / effective_load_kg
    return variable_cost_per_kg

# FUNCTION: cost per kg
def calculate_tanker_transport_cost_per_kg(
    one_way_distance_km: float,
    one_way_time_min: float,
    avg_one_way_dist_km: float
) -> dict:
    fixed_cost = calculate_tanker_fixed_cost_per_kg(avg_one_way_dist_km)
    variable_cost = calculate_tanker_variable_cost_per_kg(one_way_distance_km, one_way_time_min)
    total_cost = fixed_cost + variable_cost
    
    return {
        'fixed_cost_per_kg': fixed_cost,
        'variable_cost_per_kg': variable_cost,
        'total_cost_per_kg': total_cost
    }

Effective load 85%): 20400.0 kg
Hourly labor cost: $1.1639/h

Fixed annual costs:
  Capital (tanker): $9,199.56/y
  Licence: $91.27/y
  TOTAL: $9,290.83/y


# 8. function: assign nearest tank by traveltime

In [144]:
# Override: assign nearest tank with travel-time and real route distance in parallel
# NOTE: distance is computed along the least-time path on the friction surface.
def assign_nearest_tank_by_traveltime(facilities_gdf, storage_gdf, friction_path):
    """
    Assign nearest primary storage by travel time using OnStove friction routing.

    Adds fields:
    - tank_fid: unique storage identifier (prefer FID, fallback to index-based id)
    - tank_name: nearest storage name by minimum travel time
    - tank_traveltime: modeled travel time in minutes
    - tank_time: alias of tank_traveltime (minutes)
    - tank_distance: route distance in km along the least-time path
    """
    facilities = facilities_gdf.copy()

    from skimage.graph import MCP_Geometric
    from pyproj import Geod

    if facilities.empty:
        facilities['tank_fid'] = pd.Series(dtype='object')
        facilities['tank_name'] = pd.Series(dtype='object')
        facilities['tank_distance'] = pd.Series(dtype='float64')
        facilities['tank_traveltime'] = pd.Series(dtype='float64')
        facilities['tank_time'] = pd.Series(dtype='float64')
        return facilities

    if storage_gdf is None or storage_gdf.empty:
        facilities['tank_fid'] = None
        facilities['tank_name'] = None
        facilities['tank_distance'] = np.nan
        facilities['tank_traveltime'] = np.nan
        facilities['tank_time'] = np.nan
        return facilities

    friction_file = Path(friction_path)
    if not friction_file.exists():
        raise FileNotFoundError(f"Missing friction raster: {friction_file}")

    if facilities.crs is None:
        raise ValueError("Facilities layer has no CRS")
    if storage_gdf.crs is None:
        raise ValueError("primary_storage layer has no CRS")

    friction = RasterLayer(name='friction_surface', path=str(friction_file))
    friction_crs = friction.meta.get('crs')
    if friction_crs is None:
        raise ValueError("Friction raster has no CRS in metadata")

    facilities_tt = facilities.to_crs(friction_crs).copy()
    storage_tt = storage_gdf.to_crs(friction_crs).copy().reset_index(drop=True)

    # Build a guaranteed unique tank identifier, preferring FID-like columns when valid
    fid_col = None
    for col in ['FID', 'fid', 'id', 'id_res&fil']:
        if col in storage_tt.columns:
            fid_col = col
            break

    storage_tt['__tank_fid'] = storage_tt.index.astype(str)
    if fid_col is not None:
        fid_vals = storage_tt[fid_col].astype(str)
        invalid = fid_vals.str.strip().isin(['', 'nan', 'None'])
        duplicated = fid_vals.duplicated(keep=False)
        use_col = (~invalid) & (~duplicated)
        storage_tt.loc[use_col, '__tank_fid'] = fid_vals.loc[use_col]

    name_col = 'name' if 'name' in storage_tt.columns else None
    if name_col is None:
        for col in storage_tt.columns:
            if str(col).lower() == 'name':
                name_col = col
                break
    if name_col is None:
        storage_tt['__tank_name'] = [f'primary_storage_{i}' for i in storage_tt.index]
        name_col = '__tank_name'
    else:
        storage_tt[name_col] = storage_tt[name_col].astype(str)
        missing_names = storage_tt[name_col].str.strip().isin(['', 'nan', 'None'])
        storage_tt.loc[missing_names, name_col] = [f'primary_storage_{i}' for i in storage_tt.index[missing_names]]

    facility_points = facilities_tt.geometry
    if not facilities_tt.geom_type.eq('Point').all():
        facility_points = facilities_tt.geometry.centroid

    storage_points = storage_tt.geometry
    if not storage_tt.geom_type.eq('Point').all():
        storage_points = storage_tt.geometry.centroid

    facility_rows = []
    facility_cols = []
    for geom in facility_points:
        row, col = rasterio.transform.rowcol(friction.meta['transform'], geom.x, geom.y)
        facility_rows.append(row)
        facility_cols.append(col)

    travel_matrix = np.full((len(facilities_tt), len(storage_tt)), np.nan, dtype=np.float64)
    distance_matrix = np.full((len(facilities_tt), len(storage_tt)), np.nan, dtype=np.float64)

    # Build cost surface exactly as RasterLayer.travel_time does (hours/km)
    cost_layer = friction.data.astype(np.float64).copy()
    cost_layer *= (1000.0 / 60.0)
    nodata = friction.meta.get('nodata')
    cost_layer[np.isnan(cost_layer)] = np.inf
    if nodata is not None:
        cost_layer[cost_layer == nodata] = np.inf
    cost_layer[cost_layer < 0] = np.inf

    transform = friction.meta['transform']
    max_row, max_col = cost_layer.shape
    crs_obj = friction.meta.get('crs')
    use_geodesic = bool(getattr(crs_obj, 'is_geographic', False))
    geod = Geod(ellps='WGS84') if use_geodesic else None

    for tank_idx, tank_geom in enumerate(storage_points):
        t_row, t_col = rasterio.transform.rowcol(transform, tank_geom.x, tank_geom.y)
        if not (0 <= t_row < max_row and 0 <= t_col < max_col):
            continue

        mcp = MCP_Geometric(cost_layer, fully_connected=True)
        cumulative_costs, _ = mcp.find_costs(starts=np.array([[t_row, t_col]], dtype=np.int64))

        for fac_idx, (row, col) in enumerate(zip(facility_rows, facility_cols)):
            if 0 <= row < max_row and 0 <= col < max_col:
                val = cumulative_costs[row, col]
                if np.isfinite(val):
                    travel_matrix[fac_idx, tank_idx] = float(val)
                    path = mcp.traceback((row, col))
                    if len(path) <= 1:
                        distance_matrix[fac_idx, tank_idx] = 0.0
                        continue

                    p_rows = np.array([p[0] for p in path], dtype=np.int64)
                    p_cols = np.array([p[1] for p in path], dtype=np.int64)
                    xs, ys = rasterio.transform.xy(transform, p_rows, p_cols, offset='center')
                    xs = np.asarray(xs, dtype=np.float64)
                    ys = np.asarray(ys, dtype=np.float64)

                    if use_geodesic:
                        _, _, seg_m = geod.inv(xs[:-1], ys[:-1], xs[1:], ys[1:])
                        dist_km = float(np.nansum(np.abs(seg_m)) / 1000.0)
                    else:
                        dx = np.diff(xs)
                        dy = np.diff(ys)
                        dist_km = float(np.nansum(np.hypot(dx, dy)) / 1000.0)

                    distance_matrix[fac_idx, tank_idx] = dist_km

    tank_id_supply = np.array([None] * len(facilities_tt), dtype=object)
    tank_name = np.array([None] * len(facilities_tt), dtype=object)
    tank_distance = np.full(len(facilities_tt), np.nan, dtype=np.float64)
    tank_traveltime = np.full(len(facilities_tt), np.nan, dtype=np.float64)

    valid = np.any(np.isfinite(travel_matrix), axis=1)
    if np.any(valid):
        best_tank_idx = np.nanargmin(travel_matrix[valid], axis=1)
        valid_rows = np.where(valid)[0]

        storage_ids = storage_tt['id_supply'].astype(str).to_numpy()
        storage_names = storage_tt[name_col].astype(str).to_numpy()
        
        tank_id_supply[valid] = storage_ids[best_tank_idx]
        tank_name[valid] = storage_names[best_tank_idx]


        # OnStove travel_time output is treated as hours: convert to minutes as requested
        best_tt_hours = travel_matrix[valid_rows, best_tank_idx]
        best_tt_min = best_tt_hours * 60.0
        tank_traveltime[valid] = best_tt_min

        # Real route distance along the same least-time path
        tank_distance[valid] = distance_matrix[valid_rows, best_tank_idx]

# --- FINAL ASSIGNMENT 
    facilities['tank_id_supply'] = tank_id_supply
    facilities['tank_name'] = tank_name
    facilities['tank_distance'] = tank_distance
    facilities['tank_traveltime'] = tank_traveltime
    facilities['tank_time'] = tank_traveltime
    
    return facilities

# 9.1. apply processing, assign tanks, calculate transport cost per kg [WARNING!!!!!!!! SEE NOTE]

In [145]:
# Apply processing for each facility type
lpg_price_raster = data_dir / 'fob_per_kg_fake.tif' #NOTE - replace with actual fob_per_kg raster path
refineries = process_refineries(refineries, default_data['refineries'], lpg_price_raster_path=lpg_price_raster, price_band=1)
primary_storage = process_primary_storage(primary_storage, default_data['primary_storage'], lpg_price_raster_path=lpg_price_raster, price_band=1)
ports = process_ports(ports, default_data['ports'], primary_storage, lpg_import_cost, lpg_price_raster_path=lpg_price_raster, price_band=1)
gas_plants = process_gas_plants(gas_plants, default_data['gas_plants'], lpg_price_raster_path=lpg_price_raster, price_band=1)
border_points = process_border_points(border_points, default_data['border_points'], national_shares, lpg_import_cost, lpg_price_raster_path=lpg_price_raster, price_band=1)

# Assign nearest primary storage by travel time (OnStove)
friction_path = data_dir / 'friction_moto.tif'
refineries = assign_nearest_tank_by_traveltime(refineries, primary_storage, friction_path)
ports = assign_nearest_tank_by_traveltime(ports, primary_storage, friction_path)
gas_plants = assign_nearest_tank_by_traveltime(gas_plants, primary_storage, friction_path)
border_points = assign_nearest_tank_by_traveltime(border_points, primary_storage, friction_path)

# Compute transport_cost_to_tanker for facility layers only
all_facilities = [refineries, ports, gas_plants, border_points]
all_dist = np.concatenate([
    pd.to_numeric(gdf['tank_distance'], errors='coerce').to_numpy(dtype=float)
    for gdf in all_facilities if 'tank_distance' in gdf.columns
])
avg_one_way_dist_km = float(np.nanmean(all_dist[np.isfinite(all_dist) & (all_dist > 0)])) if np.any(np.isfinite(all_dist)) else np.nan

for gdf in all_facilities:
    if 'tank_distance' not in gdf.columns or 'tank_traveltime' not in gdf.columns:
        gdf['transport_cost_to_tanker'] = np.nan
        continue

    d_km = pd.to_numeric(gdf['tank_distance'], errors='coerce').to_numpy(dtype=float)
    t_min = pd.to_numeric(gdf['tank_traveltime'], errors='coerce').to_numpy(dtype=float)
    out_cost = np.full(len(gdf), np.nan, dtype=float)

    valid = np.isfinite(d_km) & (d_km > 0) & np.isfinite(t_min) & (t_min >= 0) & np.isfinite(avg_one_way_dist_km) & (avg_one_way_dist_km > 0)
    valid_idx = np.where(valid)[0]
    for i in valid_idx:
        out_cost[i] = calculate_tanker_transport_cost_per_kg(
            one_way_distance_km=float(d_km[i]),
            one_way_time_min=float(t_min[i]),
            avg_one_way_dist_km=avg_one_way_dist_km,
        )['total_cost_per_kg']

    gdf['transport_cost_to_tanker'] = out_cost

# Storage-level aggregation is done after percentage calculation (weighted by facility percentage)

border_points: total percentage assigned = 0.00%


# 9.2. calculate market share %

In [146]:
# STEP 4: Calculate percentages
def calculate_percentages(refineries_gdf, ports_gdf, gas_plants_gdf, border_points_gdf, nat_shares_df, storage_gdf):
    """
    CALCULATE FACILITY-LEVEL MARKET SHARE PERCENTAGES

    Computes market share by distributing national shares across individual facilities
    based on their production/storage capacity.

    border_points_gdf is expected to already contain a valid 'percentage' column
    whose sum matches the national border_points share.

    OUTPUT:
    - result: combined GeoDataFrame for quick checks/summary
    - by_category: dict with one GeoDataFrame per source category, preserving original fields
    """

    # Choose a common CRS for all facility layers before concatenation
    target_crs = refineries_gdf.crs or ports_gdf.crs or gas_plants_gdf.crs or border_points_gdf.crs

    # For refineries, use crude_capacity for proportional split
    refineries_weight_col = 'crude_capacity'

    def _numeric_series_or_zero(gdf, col):
        if col in gdf.columns:
            return pd.to_numeric(gdf[col], errors='coerce').fillna(0.0)
        return pd.Series(0.0, index=gdf.index, dtype='float64')

    # Calculate totals
    totals = {
        'refineries': _numeric_series_or_zero(refineries_gdf, refineries_weight_col).sum(),
        'ports': _numeric_series_or_zero(ports_gdf, 'LPG_capacity').sum(),
        'gas_plants': _numeric_series_or_zero(gas_plants_gdf, 'LPG_prod').sum(),
    }

    # Get national shares from raster-derived values (no fallback defaults)
    nat_shares = {
        'refineries': nat_shares_df.loc['refineries'].iloc[0] / 100,
        'ports': nat_shares_df.loc['ports'].iloc[0] / 100,
        'gas_plants': nat_shares_df.loc['gas_plants'].iloc[0] / 100,
        'border_points': nat_shares_df.loc['border_points'].iloc[0] / 100,
    }

    print(f"Refineries proportional column: {refineries_weight_col}")
    print(f"Totals: Refineries={totals['refineries']:.0f} | Ports={totals['ports']:.0f} | Gas={totals['gas_plants']:.0f}")
    print(f"Shares: {nat_shares}")

    by_category = {}
    specs = [
        ('refineries', refineries_gdf, refineries_weight_col),
        ('ports', ports_gdf, 'LPG_capacity'),
        ('gas_plants', gas_plants_gdf, 'LPG_prod'),
    ]

    for src_name, src_gdf, weight_col in specs:
        gdf = src_gdf.copy()

        # Reproject each layer to a common CRS to avoid concat CRS conflicts
        if target_crs is not None and gdf.crs is not None and gdf.crs != target_crs:
            gdf = gdf.to_crs(target_crs)

        # Ensure required columns used later in the pipeline are present
        if 'name' not in gdf.columns:
            gdf['name'] = [f'{src_name}_{idx}' for idx in gdf.index]

        if 'LPG_price' not in gdf.columns:
            gdf['LPG_price'] = np.nan
        else:
            gdf['LPG_price'] = pd.to_numeric(gdf['LPG_price'], errors='coerce')

        if 'country' not in gdf.columns:
            gdf['country'] = 'Unknown'
        else:
            gdf['country'] = gdf['country'].fillna('Unknown')

        # Keep category in each layer and compute percentage
        gdf['category'] = src_name if src_name != 'gas_plants' else 'gasplants'

        weights = _numeric_series_or_zero(gdf, weight_col)

        if totals[src_name] > 0:
            gdf['percentage'] = nat_shares[src_name] * (weights / totals[src_name])
        else:
            gdf['percentage'] = 0.0

        by_category[src_name] = gdf

    # Border points: keep precomputed percentages from processing step
    border = border_points_gdf.copy()
    if target_crs is not None and border.crs is not None and border.crs != target_crs:
        border = border.to_crs(target_crs)

    if 'name' not in border.columns:
        border['name'] = [f'border_points_{idx}' for idx in border.index]

    if 'LPG_price' not in border.columns:
        border['LPG_price'] = np.nan
    else:
        border['LPG_price'] = pd.to_numeric(border['LPG_price'], errors='coerce')

    if 'country' not in border.columns:
        border['country'] = 'Unknown'
    else:
        border['country'] = border['country'].fillna('Unknown')

    border['category'] = 'border_points'
    if 'percentage' not in border.columns:
        border['percentage'] = 0.0
    else:
        border['percentage'] = pd.to_numeric(border['percentage'], errors='coerce').fillna(0.0)

    by_category['border_points'] = border

    result = gpd.GeoDataFrame(
        pd.concat(
            [
                by_category['refineries'],
                by_category['ports'],
                by_category['gas_plants'],
                by_category['border_points'],
            ],
            ignore_index=True,
        ),
        crs=target_crs,
    )

    print(f"border_points share in result: {by_category['border_points']['percentage'].sum():.2%}")
    print(f"✓ {len(result)} sources calculated | Total share: {result['percentage'].sum():.2%}")
    return result, by_category

# Use in-memory processed data
beginning, facilities_by_category = calculate_percentages(refineries, ports, gas_plants, border_points, national_shares, primary_storage)

Refineries proportional column: crude_capacity
Totals: Refineries=446 | Ports=686125 | Gas=1445000
Shares: {'refineries': np.float64(0.0), 'ports': np.float64(0.507595971268243), 'gas_plants': np.float64(0.4924040287317571), 'border_points': np.float64(0.0)}
border_points share in result: 0.00%
✓ 26 sources calculated | Total share: 100.00%


# 10. aggregate storage transport cost (weighted by supply shares)

In [147]:
# Aggregate facility transport_cost_to_tanker on primary_storage using weighted average by facility percentage
primary_storage = primary_storage.copy()
if 'name' not in primary_storage.columns:
    primary_storage['name'] = [f'primary_storage_{i}' for i in primary_storage.index]
primary_storage['name'] = primary_storage['name'].astype(str)

weighted_cost_parts = []
for key in ['refineries', 'ports', 'gas_plants', 'border_points']:
    gdf = facilities_by_category.get(key)
    if gdf is None:
        continue
    required = {'tank_name', 'transport_cost_to_tanker', 'percentage'}
    if not required.issubset(gdf.columns):
        continue

    tmp = gdf[['tank_name', 'transport_cost_to_tanker', 'percentage']].copy()
    tmp['tank_name'] = tmp['tank_name'].astype(str)
    tmp['transport_cost_to_tanker'] = pd.to_numeric(tmp['transport_cost_to_tanker'], errors='coerce')
    tmp['percentage'] = pd.to_numeric(tmp['percentage'], errors='coerce').fillna(0.0)
    tmp = tmp[np.isfinite(tmp['transport_cost_to_tanker']) & (tmp['percentage'] > 0)]
    if len(tmp) > 0:
        weighted_cost_parts.append(tmp)

if len(weighted_cost_parts) > 0:
    weighted_df = pd.concat(weighted_cost_parts, ignore_index=True)
    weighted_df['weighted_cost'] = weighted_df['transport_cost_to_tanker'] * weighted_df['percentage']

    agg = weighted_df.groupby('tank_name', as_index=False).agg(
        weighted_cost_sum=('weighted_cost', 'sum'),
        weight_sum=('percentage', 'sum'),
    )

    agg['transport_cost_to_tanker'] = np.where(
        agg['weight_sum'] > 0,
        agg['weighted_cost_sum'] / agg['weight_sum'],
        np.nan,
    )

    primary_storage = primary_storage.merge(
        agg[['tank_name', 'transport_cost_to_tanker']],
        left_on='name',
        right_on='tank_name',
        how='left',
    )
    if 'tank_name' in primary_storage.columns:
        primary_storage = primary_storage.drop(columns=['tank_name'])
else:
    primary_storage['transport_cost_to_tanker'] = np.nan

print('✓ primary_storage transport_cost_to_tanker aggregated with percentage-weighted average')

✓ primary_storage transport_cost_to_tanker aggregated with percentage-weighted average


# 11. save initial output

In [ ]:
# STEP 5: Save output as a multi-layer GPKG (facility layers + primary storage + border points)
output_gpkg = data_dir / "beginning.gpkg"
output_gpkg_alt = data_dir / "beginning_new.gpkg"

layer_map = [
    ('refineries', 'refineries'),
    ('ports', 'ports'),
    ('gas_plants', 'gas_plants'),
    ('primary_storage', 'primary_storage'),
    ('border_points', 'border_points'),
]

export_layers = {
    'refineries': facilities_by_category['refineries'],
    'ports': facilities_by_category['ports'],
    'gas_plants': facilities_by_category['gas_plants'],
    'primary_storage': primary_storage_export,
    'border_points': facilities_by_category['border_points'],
}

def _sanitize_for_gpkg(gdf):
    """Make a GeoDataFrame safe for GPKG writing while preserving all fields."""
    cleaned = gdf.copy()

    # Avoid case-insensitive duplicate field names in GPKG
    seen = {}
    new_cols = []
    for col in cleaned.columns:
        key = str(col).lower()
        if key in seen:
            seen[key] += 1
            new_cols.append(f"{col}_{seen[key]}")
        else:
            seen[key] = 0
            new_cols.append(col)
    cleaned.columns = new_cols

    # Convert problematic object fields to strings to avoid writer errors
    geom_col = cleaned.geometry.name if hasattr(cleaned, 'geometry') else 'geometry'
    for col in cleaned.columns:
        if col == geom_col:
            continue
        if cleaned[col].dtype == 'object':
            cleaned[col] = cleaned[col].apply(
                lambda v: v if isinstance(v, (str, int, float, bool, type(None), np.integer, np.floating)) else str(v)
            )

    return cleaned


def _write_layers(target_path):
    if target_path.exists():
        target_path.unlink()
    for i, (key, layer_name) in enumerate(layer_map):
        gdf = _sanitize_for_gpkg(export_layers[key])
        mode = 'w' if i == 0 else 'a'
        gdf.to_file(target_path, layer=layer_name, driver="GPKG", mode=mode)


actual_output_gpkg = output_gpkg
try:
    _write_layers(output_gpkg)
except PermissionError:
    print(f"[warn] Cannot overwrite {output_gpkg}. Saving to {output_gpkg_alt.name} instead.")
    actual_output_gpkg = output_gpkg_alt
    _write_layers(actual_output_gpkg)

# Keep combined dataframe available for summary and quick checks
output_gdf = beginning.copy()

print(f"\n✓ beginning.gpkg export completed")
print(f"  Output file: {actual_output_gpkg}")
for key, layer_name in layer_map:
    gdf = export_layers[key]
    print(f"  - Layer '{layer_name}': {len(gdf)} records, {len(gdf.columns)} fields")
print(f"  Combined records (all layers): {len(output_gdf)}")



✓ beginning.gpkg export completed
  Output file: dataset_first_step\beginning.gpkg
  - Layer 'refineries': 4 records, 16 fields
  - Layer 'ports': 11 records, 50 fields
  - Layer 'gas_plants': 5 records, 16 fields
  - Layer 'primary_storage': 19 records, 21 fields
  - Layer 'border_points': 6 records, 36 fields
  Combined records (all layers): 26


# 12. print summary

In [149]:
# STEP 6: Summary
print("PROCESSING COMPLETE")

print(f"\nConstants: STS_cost={STS_cost}, BUFFER_KM={BUFFER_KM}")
print("LPG_price source: sampled from fob_per_kg.tif at each facility location")
print(f"\nSources by category:")
for cat, count in output_gdf.groupby('category').size().items():
    share = output_gdf[output_gdf['category'] == cat]['percentage'].sum()
    print(f"  - {cat}: {count} sources ({share:.1%} share)")

if 'country' in output_gdf.columns:
    print(f"\nSources by country:")
    for country in sorted(output_gdf['country'].dropna().unique()):
        records = output_gdf[output_gdf['country'] == country]
        print(f"  - {country}: {len(records)} sources ({records['percentage'].sum():.1%})")

print(f"\nOutputs: {data_dir}/")
if (data_dir / 'beginning.gpkg').exists():
    print("  ✓ beginning.gpkg")
if (data_dir / 'beginning_border_points.gpkg').exists():
    print("  ✓ beginning_border_points.gpkg")


PROCESSING COMPLETE

Constants: STS_cost=0.1, BUFFER_KM=10
LPG_price source: sampled from fob_per_kg.tif at each facility location

Sources by category:
  - border_points: 6 sources (0.0% share)
  - gasplants: 5 sources (49.2% share)
  - ports: 11 sources (50.8% share)
  - refineries: 4 sources (0.0% share)

Sources by country:
  - NGA: 20 sources (100.0%)
  - Unknown: 6 sources (0.0%)

Outputs: dataset_first_step/
  ✓ beginning.gpkg


# 13. assign tank to filling (demand side)

In [150]:
import geopandas as gpd

def main():
    # Define input and output paths
    filling_points_path = "dataset_big/filling_point_clients.gpkg"
    primary_storage_path = "dataset_first_step/beginning.gpkg"
    friction_raster_path = "dataset_first_step/friction_moto.tif"
    output_path = "dataset_big/filling_point_assigned.gpkg"

    # Load vector files
    facilities_gdf = gpd.read_file(filling_points_path)
    storage_gdf = gpd.read_file(primary_storage_path, layer='primary_storage')

    # Reproject filling points to match mask_layer CRS before clipping
    if facilities_gdf.crs != mask_layer.crs:
        facilities_gdf = facilities_gdf.to_crs(mask_layer.crs)

    # Clip filling points with mask layer (same as other layers)
    facilities_gdf = gpd.clip(facilities_gdf, mask_layer)
    print(f"Filling points clipped to mask_layer: {len(facilities_gdf)} records")

    # Call the pre-existing function
    result_gdf = assign_nearest_tank_by_traveltime(
        facilities_gdf=facilities_gdf,
        storage_gdf=storage_gdf,
        friction_path=friction_raster_path
    )

    # Remove legacy key if present
    if 'tank_id_res_fil' in result_gdf.columns:
        result_gdf = result_gdf.drop(columns=['tank_id_res_fil'])

    # Save the output to a new file to prevent overwriting the original data
    result_gdf.to_file(output_path, driver="GPKG")
    print("Saved columns include: tank_id_supply, tank_distance, tank_traveltime")

if __name__ == "__main__":
    main()

Filling points clipped to mask_layer: 375 records
Saved columns include: tank_id_supply, tank_distance, tank_traveltime


# 14. percentage supply and demand aggregation

In [152]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np

print("\n=== Aggregating supply and demand percentages ===")

beginning_path = data_dir / "beginning.gpkg"
filling_assigned_path = Path("dataset_big/filling_point_assigned.gpkg")

# 1. Load primary storage and ensure id_supply exists
ps = gpd.read_file(beginning_path, layer="primary_storage").copy()

# Ensure we have a clean id_supply and it's a string
if "id_supply" not in ps.columns:
    ps = ps.reset_index(drop=True)
    ps["id_supply"] = [f"primary_storage_{i}" for i in range(len(ps))]
ps["id_supply"] = ps["id_supply"].astype(str)

# Pre-initialize percentage columns to 0.0
ps["percentage_supply"] = 0.0
ps["percentage_demand"] = 0.0

# Create a mapping dictionary for names (as a fallback)
name_to_id = dict(zip(ps['name'].astype(str), ps['id_supply'].astype(str)))

# 2. Process Supply (Refineries, Ports, etc.)
supply_parts = []
for lyr in ["refineries", "ports", "gas_plants", "border_points"]:
    try:
        fac = gpd.read_file(beginning_path, layer=lyr)
        if "percentage" in fac.columns:
            tmp = fac.copy()
            # Find which tank is assigned
            if "tank_id_supply" in tmp.columns:
                tmp["join_key"] = tmp["tank_id_supply"].astype(str)
            elif "tank_name" in tmp.columns:
                tmp["join_key"] = tmp["tank_name"].astype(str).map(name_to_id)
            else: continue
            
            tmp["percentage"] = pd.to_numeric(tmp["percentage"], errors="coerce").fillna(0.0)
            supply_parts.append(tmp[["join_key", "percentage"]])
    except: continue

if supply_parts:
    supply_df = pd.concat(supply_parts, ignore_index=True)
    supply_sum = supply_df.groupby("join_key")["percentage"].sum()
    ps["percentage_supply"] = ps["id_supply"].map(supply_sum).fillna(0.0)

# 3. Process Demand (Filling Points)
filling = gpd.read_file(filling_assigned_path)
if "tank_id_supply" in filling.columns:
    filling["join_key"] = filling["tank_id_supply"].astype(str)
elif "tank_name" in filling.columns:
    filling["join_key"] = filling["tank_name"].astype(str).map(name_to_id)
else:
    raise ValueError("No tank assignment found in filling_point_assigned.gpkg")

# Determine demand column
demand_col = next((c for c in ["percentage_demand", "total_fil_clients", "clients"] if c in filling.columns), None)
if demand_col:
    demand_vals = pd.to_numeric(filling[demand_col], errors="coerce").fillna(0.0)
    if demand_col != "percentage_demand": # Convert to share if it's raw clients count
        demand_vals = demand_vals / demand_vals.sum()
    filling["demand_share"] = demand_vals
    demand_sum = filling.groupby("join_key")["demand_share"].sum()
    ps["percentage_demand"] = ps["id_supply"].map(demand_sum).fillna(0.0)

# 4. Save directly to beginning.gpkg
tmp_path = data_dir / "beginning_temp.gpkg"
if tmp_path.exists(): tmp_path.unlink()

ps.to_file(tmp_path, layer="primary_storage", driver="GPKG", mode="w")

for lyr in ["refineries", "ports", "gas_plants", "border_points"]:
    try: gpd.read_file(beginning_path, layer=lyr).to_file(tmp_path, layer=lyr, driver="GPKG", mode="a")
    except: continue

if beginning_path.exists(): beginning_path.unlink()
tmp_path.rename(beginning_path)

print(f"✓ Supply/Demand percentages aggregated and saved to {beginning_path}")


=== Aggregating supply and demand percentages ===
✓ Supply/Demand percentages aggregated and saved to dataset_first_step\beginning.gpkg


# 15. storage cost calculation

In [161]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np

print("\n=== Calculating first and second storage costs ===")

beginning_path = data_dir / "beginning.gpkg"
filling_assigned_path = Path("dataset_big/filling_point_assigned.gpkg")

if not beginning_path.exists():
    raise FileNotFoundError(f"Missing input GPKG: {beginning_path}")
if not filling_assigned_path.exists():
    raise FileNotFoundError(f"Missing input GPKG: {filling_assigned_path}")

# Read from the main file
primary_storage_sc = gpd.read_file(beginning_path, layer="primary_storage").copy()
filling_points_gdf = gpd.read_file(filling_assigned_path).copy()

# ---- Parameters and Annual Flows ----
meals_per_day = 1.0
energy_per_meal_mj = 3.64
efficiency = 0.5
energy_content_mj_per_kg = 45.5
density_kg_per_l = 0.5
kg_per_m3 = 1000.0 * density_kg_per_l

# Capacity Calculation
capacity_m3 = pd.to_numeric(primary_storage_sc.get("LPG_capacity"), errors="coerce")
avg_capacity_m3 = capacity_m3[np.isfinite(capacity_m3) & (capacity_m3 > 0)].mean()
capacity_m3 = capacity_m3.fillna(avg_capacity_m3)
capacity_kg = capacity_m3 * kg_per_m3
primary_storage_sc["used_capacity_kg"] = capacity_kg

# Total national flow (Supply = Demand = Total flow)
annual_kg_per_client = (meals_per_day * 365.0 * energy_per_meal_mj) / (efficiency * energy_content_mj_per_kg)
total_clients = pd.to_numeric(filling_points_gdf.get("total_fil_clients"), errors="coerce").fillna(0.0).sum()
total_annual_flow_kg = float(total_clients) * annual_kg_per_client

# ---- Residence Time Calculation ----
# 1. First Residence (based on supply)
pct_supply = pd.to_numeric(primary_storage_sc.get("percentage_supply"), errors="coerce").fillna(0.0)
tank_annual_supply_kg = total_annual_flow_kg * pct_supply
first_residence_years = np.where(tank_annual_supply_kg > 0, capacity_kg / tank_annual_supply_kg, np.nan)

# 2. Second Residence (based on demand)
pct_demand = pd.to_numeric(primary_storage_sc.get("percentage_demand"), errors="coerce").fillna(0.0)
tank_annual_demand_kg = total_annual_flow_kg * pct_demand
second_residence_years = np.where(tank_annual_demand_kg > 0, capacity_kg / tank_annual_demand_kg, np.nan)

# ---- Cost Function ----
def calculate_residence_cost_per_storage(
    storage_overnight_cost_usd_kg, storage_installation_fee, capacity_kg_series,
    discount_rate, storage_life_years, storage_license_cost_usd,
    storage_license_life_years, residence_time_years_series
):
    crf_capex = crf(discount_rate, storage_life_years)
    crf_license = crf(discount_rate, storage_license_life_years)

    safe_capacity_kg = capacity_kg_series.replace(0, np.nan)
    
    # All costs normalized to US$/kg of capacity
    capex_component = (storage_overnight_cost_usd_kg + (storage_installation_fee / safe_capacity_kg)) * crf_capex
    license_component = (storage_license_cost_usd / safe_capacity_kg) * crf_license

    # Final cost = (Annualized Capex + Annualized License) * Residence Time
    return (capex_component + license_component) * residence_time_years_series

# ---- Economic Constants ----
storage_overnight_cost_usd_kg = 12.5
storage_installation_fee = 760.0
storage_license_cost_usd = 3952.0
storage_life_years = 50
storage_license_life_years = 3
discount_rate_storage = float(globals().get("discount_rate", 0.03))

# ---- Cost Assignment with 0.25 CAP ----
storage_cap = 0.25

primary_storage_sc["first_storage_cost"] = calculate_residence_cost_per_storage(
    storage_overnight_cost_usd_kg, storage_installation_fee, capacity_kg,
    discount_rate_storage, storage_life_years, storage_license_cost_usd,
    storage_license_life_years, pd.Series(first_residence_years, index=primary_storage_sc.index)
).clip(upper=storage_cap)

primary_storage_sc["second_storage_cost"] = calculate_residence_cost_per_storage(
    storage_overnight_cost_usd_kg, storage_installation_fee, capacity_kg,
    discount_rate_storage, storage_life_years, storage_license_cost_usd,
    storage_license_life_years, pd.Series(second_residence_years, index=primary_storage_sc.index)
).clip(upper=storage_cap)

# ==== DIRECT SAVE TO BEGINNING.GPKG ====
tmp_path = data_dir / "beginning_temp.gpkg"
if tmp_path.exists():
    tmp_path.unlink()

primary_storage_sc.to_file(tmp_path, layer="primary_storage", driver="GPKG", mode="w")

layers_to_preserve = ["refineries", "ports", "gas_plants", "border_points"]
for layer_name in layers_to_preserve:
    try:
        out_layer = gpd.read_file(beginning_path, layer=layer_name)
        out_layer.to_file(tmp_path, layer=layer_name, driver="GPKG", mode="a")
    except Exception:
        continue

if beginning_path.exists():
    beginning_path.unlink()
tmp_path.rename(beginning_path)

print(f"✓ Storage costs saved in {beginning_path}")
print("Fields updated: used_capacity_kg, first_storage_cost, second_storage_cost")


=== Calculating first and second storage costs ===
✓ Storage costs saved in dataset_first_step\beginning.gpkg
Fields updated: used_capacity_kg, first_storage_cost, second_storage_cost


# 16. allocation optimization problem

In [162]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from scipy.optimize import linprog
from openpyxl import Workbook
from onstove.layer import VectorLayer, RasterLayer
from skimage.graph import MCP_Geometric
from pyproj import Geod

# Paths
beginning_path = data_dir / "beginning.gpkg"
friction_path = data_dir / "friction_moto.tif"
output_excel_path = data_dir / "storage_allocation.xlsx"

# Load primary storage with supply/demand percentages
ps = gpd.read_file(beginning_path, layer="primary_storage").copy()
n_storage = len(ps)
storage_ids = ps.index.tolist()

print(f"Primary storage facilities: {n_storage}")
print(f"Supply: {ps['percentage_supply'].sum():.4f}")
print(f"Demand: {ps['percentage_demand'].sum():.4f}")

# ==== 1. CALCULATE TRAVEL TIME MATRIX ====
print("\n=== Calculating travel time matrix using friction raster ===")
friction = RasterLayer(name='friction_surface', path=str(friction_path))
storage_tt = ps.to_crs(friction.meta.get('crs')).copy()

# Extract storage centroids
storage_points = storage_tt.geometry
if not storage_tt.geom_type.eq('Point').all():
    storage_points = storage_tt.geometry.centroid

# Pre-compute travel times and real route distances in parallel
travel_time_matrix = np.full((n_storage, n_storage), np.inf, dtype=np.float64)
distance_matrix = np.full((n_storage, n_storage), np.nan, dtype=np.float64)

# Build cost surface exactly as RasterLayer.travel_time does (hours/km)
cost_layer = friction.data.astype(np.float64).copy()
cost_layer *= (1000.0 / 60.0)
nodata = friction.meta.get('nodata')
cost_layer[np.isnan(cost_layer)] = np.inf
if nodata is not None:
    cost_layer[cost_layer == nodata] = np.inf
cost_layer[cost_layer < 0] = np.inf

transform = friction.meta['transform']
max_row, max_col = cost_layer.shape
crs_obj = friction.meta.get('crs')
use_geodesic = bool(getattr(crs_obj, 'is_geographic', False))
geod = Geod(ellps='WGS84') if use_geodesic else None

for i, geom_i in enumerate(storage_points):
    t_row, t_col = rasterio.transform.rowcol(transform, geom_i.x, geom_i.y)
    if not (0 <= t_row < max_row and 0 <= t_col < max_col):
        continue

    mcp = MCP_Geometric(cost_layer, fully_connected=True)
    cumulative_costs, _ = mcp.find_costs(starts=np.array([[t_row, t_col]], dtype=np.int64))
    
    storage_points_dest = storage_points
    facility_rows = []
    facility_cols = []
    for geom in storage_points_dest:
        row, col = rasterio.transform.rowcol(friction.meta['transform'], geom.x, geom.y)
        facility_rows.append(row)
        facility_cols.append(col)
    
    for j, (row, col) in enumerate(zip(facility_rows, facility_cols)):
        if 0 <= row < max_row and 0 <= col < max_col:
            val = cumulative_costs[row, col]
            if np.isfinite(val):
                travel_time_matrix[i, j] = float(val)
                path = mcp.traceback((row, col))
                if len(path) <= 1:
                    distance_matrix[i, j] = 0.0
                    continue

                p_rows = np.array([p[0] for p in path], dtype=np.int64)
                p_cols = np.array([p[1] for p in path], dtype=np.int64)
                xs, ys = rasterio.transform.xy(transform, p_rows, p_cols, offset='center')
                xs = np.asarray(xs, dtype=np.float64)
                ys = np.asarray(ys, dtype=np.float64)

                if use_geodesic:
                    _, _, seg_m = geod.inv(xs[:-1], ys[:-1], xs[1:], ys[1:])
                    dist_km = float(np.nansum(np.abs(seg_m)) / 1000.0)
                else:
                    dx = np.diff(xs)
                    dy = np.diff(ys)
                    dist_km = float(np.nansum(np.hypot(dx, dy)) / 1000.0)

                distance_matrix[i, j] = dist_km

print(f"Travel time matrix calculated: shape {travel_time_matrix.shape}")

# ==== 2. CALCULATE DISTANCE MATRIX (ROUTE-BASED) ====
print("\n=== Calculating route-based distance matrix ===")
# distance_matrix has been computed from least-time path geometry

print(f"Distance matrix calculated: shape {distance_matrix.shape}")
print("Distance derived from least-time path geometry on friction surface")

# ==== 3. OPTIMIZE ALLOCATION (Linear Programming) ====
print("\n=== Solving transport optimization problem ===")
# Flatten allocation matrix as decision variables: x[i,j] = supply from i to j
c = travel_time_matrix.flatten()  # Minimize travel time

# Constraints:
# Supply constraints: sum(x[i,:]) = supply[i] for each i
# Demand constraints: sum(x[:,j]) = demand[j] for each j

A_eq = []
b_eq = []

# Supply constraints
for i in range(n_storage):
    row = np.zeros(n_storage * n_storage)
    row[i * n_storage:(i+1) * n_storage] = 1
    A_eq.append(row)
    b_eq.append(float(ps.iloc[i]['percentage_supply']))

# Demand constraints
for j in range(n_storage):
    row = np.zeros(n_storage * n_storage)
    for i in range(n_storage):
        row[i * n_storage + j] = 1
    A_eq.append(row)
    b_eq.append(float(ps.iloc[j]['percentage_demand']))

A_eq = np.array(A_eq)
b_eq = np.array(b_eq)

# Bounds: 0 <= x[i,j] <= max(supply[i], demand[j])
bounds = [(0, max(ps['percentage_supply'].max(), ps['percentage_demand'].max())) for _ in range(n_storage * n_storage)]

# Solve
result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

if result.success:
    allocation_flat = result.x
    allocation_matrix = allocation_flat.reshape((n_storage, n_storage))
    print(f"✓ Optimization successful")
    print(f"  Objective (total travel time): {result.fun:.4f} h")
else:
    print(f"✗ Optimization failed: {result.message}")
    allocation_matrix = np.zeros((n_storage, n_storage))

# ==== 4. WRITE EXCEL ====
print("\n=== Writing Excel ===")
wb = Workbook()
wb.remove(wb.active)  # Remove default sheet

# Storage indices as row/column labels
storage_labels = [f"{i}" for i in storage_ids]

# Sheet 1: Travel Time
ws1 = wb.create_sheet("storage_allocation_time")
ws1.append([""] + storage_labels)
for i, label_i in enumerate(storage_labels):
    row_data = [label_i] + travel_time_matrix[i, :].tolist()
    ws1.append(row_data)

# Sheet 2: Distance
ws2 = wb.create_sheet("storage_allocation_distance")
ws2.append([""] + storage_labels)
for i, label_i in enumerate(storage_labels):
    row_data = [label_i] + distance_matrix[i, :].tolist()
    ws2.append(row_data)

# Sheet 3: Allocation Percentage
ws3 = wb.create_sheet("storage_allocation_percentage")
ws3.append([""] + storage_labels)
for i, label_i in enumerate(storage_labels):
    row_data = [label_i] + allocation_matrix[i, :].tolist()
    ws3.append(row_data)

# Save with fallback if target file is open/locked
actual_output_excel = output_excel_path
wb.save(output_excel_path)


print(f"✓ Saved: {actual_output_excel}")

# Validation
print("\n=== Validation ===")
allocation_supply = allocation_matrix.sum(axis=1)
allocation_demand = allocation_matrix.sum(axis=0)
print(f"Supply check (first 5): {allocation_supply[:5]}")
print(f"Expected supply (first 5): {ps['percentage_supply'].values[:5]}")
print(f"Demand check (first 5): {allocation_demand[:5]}")
print(f"Expected demand (first 5): {ps['percentage_demand'].values[:5]}")
print(f"Total allocation: {allocation_matrix.sum():.6f}")
print(f"Total supply: {ps['percentage_supply'].sum():.6f}")
print(f"Total demand: {ps['percentage_demand'].sum():.6f}")

Primary storage facilities: 19
Supply: 1.0000
Demand: 1.0000

=== Calculating travel time matrix using friction raster ===
Travel time matrix calculated: shape (19, 19)

=== Calculating route-based distance matrix ===
Distance matrix calculated: shape (19, 19)
Distance derived from least-time path geometry on friction surface

=== Solving transport optimization problem ===
✓ Optimization successful
  Objective (total travel time): 3.6927 h

=== Writing Excel ===
✓ Saved: dataset_first_step\storage_allocation.xlsx

=== Validation ===
Supply check (first 5): [0.38267801 0.04762099 0.         0.         0.16286127]
Expected supply (first 5): [0.38267801 0.04762099 0.         0.         0.16286127]
Demand check (first 5): [0.13250231 0.         0.         0.04100374 0.02449163]
Expected demand (first 5): [0.13250231 0.         0.         0.04100374 0.02449163]
Total allocation: 1.000000
Total supply: 1.000000
Total demand: 1.000000


# 17. calculate allocation rebalancing cost

In [163]:
# This cell calculates the shipping cost associated with the allocation process
# and saves it as 'rebalancing_cost' in the latest primary storage file.

print("\n=== Calculating rebalancing costs for allocation ===")

# 1. Calculate the specific average distance for active routes (Mode 2)
# We consider only the routes where LPG is actually moved (allocation > 0)
# and ignore self-loops (distance = 0)
active_routes_mask = (allocation_matrix > 0) & (distance_matrix > 0)
valid_distances = distance_matrix[active_routes_mask]

if len(valid_distances) > 0:
    avg_dist_rebalancing_km = float(np.mean(valid_distances))
else:
    avg_dist_rebalancing_km = np.nan

print(f"Calculated average distance for rebalancing routes: {avg_dist_rebalancing_km:.2f} km")

# 2. Calculate fixed cost (Mode 2 logic: derived from the average of active routes)
if np.isfinite(avg_dist_rebalancing_km) and avg_dist_rebalancing_km > 0:
    # Uses the previously defined tanker parameters and CRF function
    fixed_cost_per_kg = calculate_tanker_fixed_cost_per_kg(avg_dist_rebalancing_km)
else:
    fixed_cost_per_kg = 0.0

# 3. Calculate weighted average cost for each destination storage
rebalancing_cost = np.full(n_storage, np.nan, dtype=np.float64)

for j in range(n_storage):
    # For each destination storage j, identify all origin storages i
    senders = np.where(allocation_matrix[:, j] > 0)[0]
    
    if len(senders) == 0:
        continue
        
    total_weighted_cost = 0.0
    total_weight = 0.0
    
    for i in senders:
        dist_km = distance_matrix[i, j]
        time_min = travel_time_matrix[i, j]
        weight = allocation_matrix[i, j]
        
        # If origin and destination are the same, transport cost is zero
        if i == j or dist_km == 0:
            cost_ij = 0.0
        else:
            var_cost = calculate_tanker_variable_cost_per_kg(dist_km, time_min)
            cost_ij = fixed_cost_per_kg + var_cost
            
        total_weighted_cost += cost_ij * weight
        total_weight += weight
        
    # Weighted average: divide by the sum of weights (demand percentage of j)
    if total_weight > 0:
        rebalancing_cost[j] = total_weighted_cost / total_weight

# 4. Add result to the primary storage DataFrame
ps['rebalancing_cost'] = rebalancing_cost

# SAVE UPDATED LAYER TO BEGINNING.GPKG 
tmp_path = data_dir / "beginning_temp.gpkg"
if tmp_path.exists(): tmp_path.unlink()

ps.to_file(tmp_path, layer="primary_storage", driver="GPKG", mode="w")

layers_to_preserve = ["refineries", "ports", "gas_plants", "border_points"]
for layer_name in layers_to_preserve:
    try:
        out_layer = gpd.read_file(beginning_path, layer=layer_name)
        out_layer.to_file(tmp_path, layer=layer_name, driver="GPKG", mode="a")
    except Exception: continue

if beginning_path.exists(): beginning_path.unlink()
tmp_path.rename(beginning_path)
print(f"✓ 'rebalancing_cost' saved to {beginning_path}")


=== Calculating rebalancing costs for allocation ===
Calculated average distance for rebalancing routes: 547.15 km
✓ 'rebalancing_cost' saved to dataset_first_step\beginning.gpkg


# 18. allocation of first storage cost

In [166]:
# This cell sums first_storage_cost and transport_cost_to_tanker, 
# then allocates the total using the allocation matrix.

import geopandas as gpd
import pandas as pd
import numpy as np

print("\n=== Calculating allocated pre_rebalancing_cost ===")
beginning_path = data_dir / "beginning.gpkg"
ps = gpd.read_file(beginning_path, layer="primary_storage").copy()
n_storage = len(ps)

# 1. Sum base costs before allocation
# We sum the local storage cost and the transport cost from sources
f_cost = pd.to_numeric(ps.get('first_storage_cost', 0.0), errors='coerce').fillna(0.0)
t_cost = pd.to_numeric(ps.get('transport_cost_to_tanker', 0.0), errors='coerce').fillna(0.0)
base_costs = (f_cost + t_cost).values

allocated_costs = np.full(n_storage, np.nan, dtype=np.float64)

# 2. Perform allocation using the matrix
for j in range(n_storage):
    senders = np.where(allocation_matrix[:, j] > 0)[0]
    if len(senders) == 0: continue
        
    total_weighted_cost = 0.0
    total_weight = 0.0
    for i in senders:
        weight = allocation_matrix[i, j]
        total_weighted_cost += base_costs[i] * weight
        total_weight += weight
        
    if total_weight > 0:
        allocated_costs[j] = total_weighted_cost / total_weight

# 3. Assign to new column name
ps['pre_rebalancing_cost_allocated'] = allocated_costs

# Remove old column if it exists to keep the file clean
if 'first_storage_cost_allocated' in ps.columns:
    ps = ps.drop(columns=['first_storage_cost_allocated'])

# 4. Save
tmp_path = data_dir / "beginning_temp.gpkg"
if tmp_path.exists(): tmp_path.unlink()
ps.to_file(tmp_path, layer="primary_storage", driver="GPKG", mode="w")
for lyr in ["refineries", "ports", "gas_plants", "border_points"]:
    try: gpd.read_file(beginning_path, layer=lyr).to_file(tmp_path, layer=lyr, driver="GPKG", mode="a")
    except: continue
if beginning_path.exists(): beginning_path.unlink()
tmp_path.rename(beginning_path)

print(f"✓ 'pre_rebalancing_cost_allocated' saved in {beginning_path}")


=== Calculating allocated pre_rebalancing_cost ===
✓ 'pre_rebalancing_cost_allocated' saved in dataset_first_step\beginning.gpkg


# 19. calculate weighted average LPG_price at each storage

In [167]:
# 19
# Sums the allocated source price with all allocated and local logistical costs.

print("\n=== Calculating final accumulated LPG_price for primary storage ===")
beginning_path = data_dir / "beginning.gpkg"
ps = gpd.read_file(beginning_path, layer="primary_storage").copy()
n_storage = len(ps)

# 1. Source Price Allocation (as previously defined)
weighted_price_parts = []
supply_layers = ['refineries', 'ports', 'gas_plants', 'border_points']
name_to_id = dict(zip(ps['name'].astype(str), ps['id_supply'].astype(str)))

for lyr in supply_layers:
    try:
        gdf = gpd.read_file(beginning_path, layer=lyr)
        if 'percentage' in gdf.columns:
            tmp = gdf.copy()
            jk = 'tank_id_supply' if 'tank_id_supply' in tmp.columns else 'tank_name'
            tmp['id_join'] = tmp[jk].astype(str).map(name_to_id) if jk == 'tank_name' else tmp[jk].astype(str)
            tmp['LPG_price'] = pd.to_numeric(tmp['LPG_price'], errors='coerce')
            tmp['percentage'] = pd.to_numeric(tmp['percentage'], errors='coerce').fillna(0.0)
            weighted_price_parts.append(tmp[np.isfinite(tmp['LPG_price']) & (tmp['percentage'] > 0)])
    except: continue

source_at_origin = np.zeros(n_storage)
if weighted_price_parts:
    combined = pd.concat(weighted_price_parts, ignore_index=True)
    combined['w_v'] = combined['LPG_price'] * combined['percentage']
    agg = combined.groupby('id_join').agg(s_w=('w_v','sum'), s_p=('percentage','sum'))
    source_at_origin = ps['id_supply'].map(agg['s_w']/agg['s_p']).fillna(0.0).values

allocated_source = np.zeros(n_storage)
for j in range(n_storage):
    senders = np.where(allocation_matrix[:, j] > 0)[0]
    if len(senders) == 0: continue
    w_p, w_t = 0.0, 0.0
    for i in senders:
        weight = allocation_matrix[i, j]
        w_p += source_at_origin[i] * weight
        w_t += weight
    if w_t > 0: allocated_source[j] = w_p / w_t

ps['LPG_source_price'] = allocated_source

# 2. Final Summation (using the new allocated column name)
ps['LPG_price'] = (
    ps['LPG_source_price'] + 
    ps.get('pre_rebalancing_cost_allocated', 0.0).fillna(0.0) + 
    ps.get('rebalancing_cost', 0.0).fillna(0.0) + 
    ps.get('second_storage_cost', 0.0).fillna(0.0)
)

# 3. Save
tmp_path = data_dir / "beginning_temp.gpkg"
if tmp_path.exists(): tmp_path.unlink()
ps.to_file(tmp_path, layer="primary_storage", driver="GPKG", mode="w")
for ln in ["refineries", "ports", "gas_plants", "border_points"]:
    try: gpd.read_file(beginning_path, layer=ln).to_file(tmp_path, layer=ln, driver="GPKG", mode="a")
    except: continue
if beginning_path.exists(): beginning_path.unlink()
tmp_path.rename(beginning_path)

print("✓ Final LPG_price accumulation complete.")


=== Calculating final accumulated LPG_price for primary storage ===
✓ Final LPG_price accumulation complete.


# 20. transport cost to filling points + new price

In [168]:
# 20.
# This cell calculates the final transport cost from storage to filling points
# and sums it with the storage LPG_price to get the landed cost at the demand point.

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

print("\n=== Calculating transport costs and final LPG_price for filling points ===")

# Paths
beginning_path = data_dir / "beginning.gpkg"
filling_assigned_path = Path("dataset_big/filling_point_assigned.gpkg")

if not beginning_path.exists() or not filling_assigned_path.exists():
    raise FileNotFoundError("Missing required input files (beginning.gpkg or filling_point_assigned.gpkg)")

# 1. Load data
ps = gpd.read_file(beginning_path, layer="primary_storage")[['id_supply', 'LPG_price']].copy()
ps = ps.rename(columns={'LPG_price': 'storage_exit_price'})
ps['id_supply'] = ps['id_supply'].astype(str)

filling = gpd.read_file(filling_assigned_path).copy()
filling['tank_id_supply'] = filling['tank_id_supply'].astype(str)

# 2. Calculate average distance specific to this leg (Storage -> Filling Point)
# This is required for the fixed cost component of the tanker function
d_km = pd.to_numeric(filling['tank_distance'], errors='coerce').to_numpy(dtype=float)
t_min = pd.to_numeric(filling['tank_traveltime'], errors='coerce').to_numpy(dtype=float)

valid_mask = np.isfinite(d_km) & (d_km > 0)
avg_dist_leg2_km = float(np.mean(d_km[valid_mask])) if np.any(valid_mask) else 0.0
print(f"Average distance for storage-to-filling routes: {avg_dist_leg2_km:.2f} km")

# 3. Calculate transport cost per kg
transport_costs = np.full(len(filling), np.nan, dtype=float)

if avg_dist_leg2_km > 0:
    for i in range(len(filling)):
        if valid_mask[i] and np.isfinite(t_min[i]):
            # Use the existing function from Cell 7
            res = calculate_tanker_transport_cost_per_kg(
                one_way_distance_km=float(d_km[i]),
                one_way_time_min=float(t_min[i]),
                avg_one_way_dist_km=avg_dist_leg2_km
            )
            transport_costs[i] = res['total_cost_per_kg']

filling['transport_cost_from_storage'] = transport_costs

# 4. Join with primary_storage to get the source price and calculate total
filling = filling.merge(
    ps, 
    left_on='tank_id_supply', 
    right_on='id_supply', 
    how='left'
).drop(columns=['id_supply'])

# Final landed cost = Price at storage exit + Transport cost to filling point
# Fillna(0) for safety in summation
exit_p = filling['storage_exit_price'].fillna(0.0)
trans_p = filling['transport_cost_from_storage'].fillna(0.0)

filling['LPG_price'] = exit_p + trans_p

# 5. Save the updated filling point layer
filling.to_file(filling_assigned_path, driver="GPKG")

print(f"✓ Transport costs and final LPG_price saved in {filling_assigned_path}")
print(f"Fields added/updated: transport_cost_from_storage, LPG_price")


=== Calculating transport costs and final LPG_price for filling points ===
Average distance for storage-to-filling routes: 159.69 km
✓ Transport costs and final LPG_price saved in dataset_big\filling_point_assigned.gpkg
Fields added/updated: transport_cost_from_storage, LPG_price
